# AlphaFold2 Structure Prediction

This notebook demonstrates protein structure prediction using AlphaFold2 via the ColabDesign JAX wrapper. We cover three common workflows with `run_alphafold2`: single-sequence prediction without MSA for fast screening, multi-chain complex prediction using the Multimer extension, and MSA-assisted prediction that leverages evolutionary information from ColabFold search for higher accuracy.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("alphafold2")
display_overview("alphafold2")
display_docs_section("alphafold2", "Background")

# AlphaFold2

AlphaFold2 (AF2) is Google DeepMind's second-generation AlphaFold, widely regarded as the first method to predict protein structures from sequence at near-experimental accuracy. This toolkit runs AF2 through [ColabDesign](https://github.com/sokrypton/ColabDesign), the open-source JAX implementation from [Sergey Ovchinnikov's group](https://www.solab.org/), to predict 3D structures of proteins and complexes and to score and differentiate binders against a fixed target.

AlphaFold2 ([Jumper et al., 2021](https://doi.org/10.1038/s41586-021-03819-2)) predicts a protein's 3D structure from its amino-acid sequence, and was introduced at the CASP14 structure-prediction assessment in 2020. AF2 takes a multiple-sequence alignment (MSA) as a primary input. The MSA carries an evolutionary signal: residues that lie close together in the folded structure tend to mutate in a correlated way across related proteins. AF2 reads these covariation patterns to infer which parts of the chain are in contact. Because the signal comes from the alignment itself, accuracy scales with the depth and diversity of the MSA, and proteins with few detectable homologs are harder to fold.

Internally, AlphaFold2 maintains two representations: an MSA representation and a pairwise representation over residue pairs. The Evoformer network repeatedly exchanges information between the two, using attention together with triangle-based updates on the pairwise representation that enforce geometric consistency among the inferred residue-residue distances. A structure module then turns these representations into an explicit 3D model, placing each residue as a rigid backbone frame with its own position and orientation. This whole process is recycled through the network several times, each pass refining the previous prediction. Along with the coordinates, AlphaFold2 emits two calibrated confidence measures: the per-residue predicted local distance difference test (pLDDT), which scores the model's confidence in each residue's local structure, and the predicted aligned error (PAE), which estimates the expected error in one residue's position when the structure is aligned on another.

This toolkit runs the original AlphaFold2 model through the [ColabDesign](https://github.com/sokrypton/ColabDesign) JAX implementation rather than the full DeepMind or ColabFold pipeline. There is no template-search stage, and multiple-sequence alignments are optional: they can be generated by a ColabFold search, supplied precomputed, or skipped to run in single-sequence mode. Beyond folding, the same model exposes a per-residue gradient, which gradient-based binder-design methods use to optimize a binder sequence against a frozen target.

## Available tools

In [2]:
display_available_tools("alphafold2")

- **`run_alphafold2_gradient()`** — AF2 binder design against a fixed target. Returns loss, Structure, and optionally gradient.
- **`run_alphafold2()`** — Protein structure prediction using AlphaFold2 via ColabDesign

### `run_alphafold2`

AlphaFold2 predicts protein 3D structures from amino acid sequences using a deep learning architecture that combines multiple sequence alignments, pairwise distance representations, and recycling iterations to generate near-experimental-accuracy coordinates. The model supports single-chain proteins and multi-chain complexes via the Multimer extension. Optional MSA generation via ColabFold search provides evolutionary context from homologous sequences, substantially improving confidence scores. The `num_recycles` parameter controls how many times predictions are iteratively refined, and the `model_num` parameter selects among five independently trained parameter sets.

In [3]:
from pathlib import Path
from proto_tools import AlphaFold2Input, AlphaFold2Config, run_alphafold2

In [4]:
# Display input docs
display_api_reference("alphafold2", "input", "run_alphafold2")

# Aequorea victoria GFP (UniProt P42212) — 238-residue β-barrel fluorescent
# protein, classic folding benchmark with dense MSA coverage. Reused below
# in the MSA-assisted example.
protein_sequence = (
    "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF"
    "ICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGY"
    "VQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILG"
    "HKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQ"
    "NTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGIT"
    "HGMDELYK"
)
inputs = AlphaFold2Input(complexes=[protein_sequence])

**Input** — `AlphaFold2Input`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `complexes` | `list[proto_tools.entities.complex.Complex]` | required | List of complexes to predict structure for containing chains and entity types. |
| `msas` | `list[proto_tools.tools.structure_prediction.shared_data_models.ComplexMSAs] \| None` | `None` | Per-complex MSAs; a bare dict[int, MSA] is coerced to an unpaired ComplexMSAs. |

In [5]:
# Display config docs
display_api_reference("alphafold2", "config", "run_alphafold2")

# Configure AlphaFold2
config = AlphaFold2Config(
    use_msa=True,
    num_recycles=3,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `AlphaFold2Config`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on (e.g., 'cuda', 'cpu') |
| `timeout` | `int \| None` | `600` | Maximum execution time in seconds. None waits indefinitely. |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `include_pae_matrix` | `bool` | `False` | Attach the full per-residue PAE matrix. |
| `use_msa` | `bool` | `True` | Whether to generate and use MSAs for protein chains using ColabFold search |
| `colabfold_search_config` | `proto_tools.tools.sequence_alignment.colabfold_search.colabfold_search.ColabfoldSearchConfig \| None` | `None` | Nested configuration for ColabFold MSA search. If None, uses default settings. |
| `num_recycles` | `int` | `3` | Recycling iterations through the model. Higher = more accurate but slower. |
| `model_num` | `int` | `1` | Which of AlphaFold2's 5 trained parameter sets to use. |
| `num_ensemble_models` | `int` | `1` | Number of parameter sets to run and average. Higher = more accurate but slower. |

In [6]:
# Run structure prediction
result = run_alphafold2(inputs, config)

Running run_colabfold_search [00:00]

Folding structures (AlphaFold2):   0%|          | 0/1 [00:00<?, ?structure/s]

In [7]:
# Display output docs
display_api_reference("alphafold2", "output", "run_alphafold2")

structure = result.structures[0]

# Print confidence metrics
print(f"  Length: {structure.num_residues} residues")
print(f"  Average pLDDT: {structure.metrics.avg_plddt:.3f}")
print(f"  pTM score: {structure.metrics.ptm:.3f}")

**Output** — `AlphaFold2Output`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `structures` | `list[proto_tools.entities.structures.structure.Structure]` | required | List of predicted structures, one per input complex. |

  Length: 238 residues
  Average pLDDT: 0.947
  pTM score: 0.889


> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [8]:
# Visualize predicted structure (auto-coloured by pLDDT)
structure.visualize()

#### Multi-chain complex prediction

When a complex contains more than one chain, AlphaFold2 automatically uses the Multimer model parameters. The iPTM score provides an additional measure of interface quality beyond the global pTM, which is especially useful for evaluating whether the predicted inter-chain contacts are reliable.

In [9]:
# Two-chain protein complex
chain_a = "MARFLGLGARYTWM"
chain_b = "YTWHKLARFGMVLS"

# Create multi-chain input
inputs_complex = AlphaFold2Input(complexes=[[chain_a, chain_b]])

# Configure (multimer mode is automatic when >1 chain)
config_complex = AlphaFold2Config(
    use_msa=False,
    num_recycles=3,
    device="cuda",  # Change to "cpu" if no GPU available
)

# Run prediction
result_complex = run_alphafold2(inputs_complex, config_complex)
complex_structure = result_complex.structures[0]

# Print metrics (iPTM is available for multimer)
print(f"  Chains: {complex_structure.num_chains}")
print(f"  Average pLDDT: {complex_structure.metrics.avg_plddt:.3f}")
print(f"  pTM score: {complex_structure.metrics.ptm:.3f}")
print(f"  iPTM score: {complex_structure.metrics.get('iptm', 'N/A')}")

# Visualize the complex with each chain in a distinct colour
complex_structure.visualize(color_by="chain")

Folding structures (AlphaFold2):   0%|          | 0/1 [00:00<?, ?structure/s]

  Chains: 2
  Average pLDDT: 0.442
  pTM score: 0.113
  iPTM score: 0.018212012946605682


## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD. The B-factor column in exported files contains normalized pLDDT confidence scores (0.0 to 1.0), so coloring by B-factor in a viewer directly shows model confidence per residue.

In [10]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to PDB files
result.export(name="alphafold2_structure", export_path=output_dir, file_format="pdb")
print(f"Structure exported to {output_dir / 'alphafold2_structure.pdb'}")

Structure exported to example_output/alphafold2_structure.pdb
